# Face 02 · Detection + 5-point Alignment

**Detector:** **YuNet** (OpenCV Zoo, MIT licence, 75 k params, gives 5 landmarks) – runs on Android/iOS as well.
Alternative: MediaPipe BlazeFace (Apache-2.0).
**Alignment:** similarity transform of the 5 landmarks to the standard ArcFace 112×112 template.
The same function must be used in training, in the apps, and on the server – or accuracy drops.

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Face_Recognition
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import cv2, numpy as np, urllib.request

YUNET = DATA / "Faces" / "face_detection_yunet_2023mar.onnx"
if not YUNET.exists():
    urllib.request.urlretrieve(
        "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx", YUNET)

ARCFACE_TEMPLATE = np.array([[38.2946, 51.6963], [73.5318, 51.5014], [56.0252, 71.7366],
                             [41.5493, 92.3655], [70.7299, 92.2041]], dtype=np.float32)

def detect(image_bgr, score=0.8):
    h, w = image_bgr.shape[:2]
    det = cv2.FaceDetectorYN.create(str(YUNET), "", (w, h), score_threshold=score)
    _, faces = det.detect(image_bgr)
    return [] if faces is None else faces            # each: x,y,w,h, 5x(x,y), score

def align(image_bgr, face_row, size=112):
    landmarks = face_row[4:14].reshape(5, 2).astype(np.float32)
    M, _ = cv2.estimateAffinePartial2D(landmarks, ARCFACE_TEMPLATE, method=cv2.LMEDS)
    return cv2.warpAffine(image_bgr, M, (size, size), borderValue=0)

In [ ]:
# Align a raw identity folder tree into Data/Faces/Train
from Common_Utils.Data_Utils import list_images
RAW = DATA / "Faces" / "Raw"
def align_tree(src=RAW, dst=DATA / "Faces" / "Train"):
    kept = 0
    for path in list_images(src):
        img = cv2.imread(str(path))
        faces = detect(img) if img is not None else []
        if len(faces) != 1:
            continue                      # skip images with 0 or several faces
        out = dst / path.parent.name / path.name
        out.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(out), align(img, faces[0]))
        kept += 1
    return kept
# align_tree()